# Issue Salience and Partisan Electoral Outcomes
## Using BERTopic, VADER, and Zero-Shot Stance Classification

**Research Question:** To what extent do state-level issue salience profiles — defined by the dominant policy concerns driving voter behavior — predict partisan electoral outcomes across U.S. states in presidential elections?

**Theoretical Framework:** Issue Ownership Theory (Petrocik, 1996)

**Methods:**
1. **BERTopic** — Unsupervised topic modeling to identify policy issues in political tweets
2. **VADER** — Lexicon-based sentiment analysis for directional issue salience
3. **Zero-Shot Classification** — Transformer-based partisan stance detection
4. **Predictive Modeling** — Logistic Regression, Random Forest, Gradient Boosting
5. **Systematic Bias Audit** — Geographic, engagement, temporal, and fairness analysis

In [1]:
# Install dependencies (Colab-compatible)
!pip install -q datasets bertopic sentence-transformers vaderSentiment transformers torch scikit-learn plotly gensim umap-learn hdbscan tqdm kaleido huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import json
import warnings
from tqdm import tqdm

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import LeaveOneOut, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
SAMPLE_SIZE = 75000
np.random.seed(RANDOM_STATE)

# Detect GPU
import torch
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device for transformers: {DEVICE}")

GPU available: True
GPU: Tesla T4
Device for transformers: 0


## 1. Data Loading and Preprocessing

Data is loaded from HuggingFace: `Diogo2110/sem4data`. The dataset contains ~2.1M tweets collected during the 2024 U.S. election cycle, with metadata including user engagement metrics, state-level geolocation, and timestamps.

In [3]:
from huggingface_hub import hf_hub_download

# Download the raw CSV directly (avoids Arrow schema parsing errors)
csv_path = hf_hub_download(
    repo_id="Diogo2110/sem4data",
    repo_type="dataset",
    filename="subjects_AND_sampling_metadata_anonymized_full.csv"
)

df = pd.read_csv(csv_path, low_memory=False)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head(3)

subjects_AND_sampling_metadata_anonymize(…):   0%|          | 0.00/3.40G [00:00<?, ?B/s]

Dataset shape: (2147714, 75)
Columns: 75


,created_at.users,protected,verified_type,public_metrics.followers_count,public_metrics.following_count,public_metrics.tweet_count,public_metrics.listed_count,public_metrics.like_count.users,public_metrics.media_count,entities.url.urls,...,clean...state_simple,user_id_code,tweet_id_code,tweet_id_historical_code,most_recent_tweet_id_code,pinned_tweet_id_code,conversation_id_code,conversation_id_historical_code,in_reply_to_user_id_code,in_reply_to_user_id_historical_code
0,2019-01-05T12:46:57.000Z,False,none,4426,1240,96732,33,208601,NaN,start.0;end.23;url.https://t.co/9oR3DiGSIZ;exp...,...,USA,uid_3120b6e3,tweet_c1890821,tweet_c1890821,tweet_c1890821,tweet_25852ced,conv_c1890821,conv_c1890821,NaN,NaN
1,2019-01-05T12:46:57.000Z,False,none,4426,1240,96732,33,208601,NaN,start.0;end.23;url.https://t.co/9oR3DiGSIZ;exp...,...,USA,uid_3120b6e3,tweet_c1890821,tweet_47d75442,tweet_c1890821,tweet_25852ced,conv_c1890821,conv_47d75442,NaN,NaN
2,2019-01-05T12:46:57.000Z,False,none,4426,1240,96732,33,208601,NaN,start.0;end.23;url.https://t.co/9oR3DiGSIZ;exp...,...,USA,uid_3120b6e3,tweet_c1890821,tweet_acbb431b,tweet_c1890821,tweet_25852ced,conv_c1890821,conv_a8a81cfe,NaN,reply_uid_ec1d751f


In [4]:
df = df.rename(columns={"clean...state_simple": "state"})

tweets_col = 'tweets_historical'

print(f"Total rows: {len(df)}")
print(f"Missing tweets: {df[tweets_col].isna().sum()}")
print(f"Missing states: {df['state'].isna().sum()}")
print(f"\nState distribution (top 10):")
print(df['state'].value_counts().head(10))

Total rows: 2147714
Missing tweets: 0
Missing states: 0

State distribution (top 10):
state
USA             439679
California      301804
Florida         256807
New York        129278
Texas           123919
New Jersey       80505
Washington       55275
Pennsylvania     55084
Ohio             54143
Colorado         51149
Name: count, dtype: int64


### Text Filtering and Cleaning

In [5]:
df = df.dropna(subset=[tweets_col])
df = df[df[tweets_col].str.strip() != '']
df = df[~df[tweets_col].str.startswith("RT @")]
df = df[df[tweets_col].str.len() >= 20]
print(f"After filtering: {len(df)} tweets")

After filtering: 2081342 tweets


In [6]:
def clean_tweet(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["tweet_clean"] = df[tweets_col].apply(clean_tweet)
df = df[df["tweet_clean"].str.len() >= 15]
print(f"After cleaning: {len(df)} tweets")

After cleaning: 1779948 tweets


### Stratified Sampling

We use a political keyword pre-filter to ensure our sample has high political signal density. Half the sample comes from tweets containing political keywords, and half from general tweets. This reduces the noise cluster problem from the baseline model.

In [7]:
political_keywords = [
    'trump', 'biden', 'harris', 'kamala', 'democrat', 'republican', 'gop',
    'election', 'vote', 'voting', 'ballot', 'congress', 'senate', 'house',
    'policy', 'tax', 'taxes', 'immigration', 'immigrant', 'border',
    'abortion', 'gun', 'guns', 'climate', 'healthcare', 'health care',
    'economy', 'inflation', 'prices', 'jobs',
    'ukraine', 'israel', 'gaza', 'hamas', 'war',
    'maga', 'liberal', 'conservative', 'progressive',
    'supreme court', 'amendment', 'legislation', 'governor', 'president',
    'debate', 'campaign', 'rally', 'poll', 'polls',
    'democrat', 'republican', 'dnc', 'rnc', 'potus',
    'ice', 'deportation', 'welfare', 'social security', 'medicare'
]

pattern = '|'.join(political_keywords)
df['is_political'] = df['tweet_clean'].str.lower().str.contains(pattern, na=False)

n_political = df['is_political'].sum()
n_general = (~df['is_political']).sum()
print(f"Political tweets: {n_political:,}")
print(f"General tweets: {n_general:,}")

half = SAMPLE_SIZE // 2
df_pol = df[df['is_political']].sample(n=min(half, n_political), random_state=RANDOM_STATE)
df_gen = df[~df['is_political']].sample(n=min(half, n_general), random_state=RANDOM_STATE)
df_sample = pd.concat([df_pol, df_gen]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

docs = df_sample["tweet_clean"].tolist()
print(f"\nFinal sample size: {len(df_sample)}")
print(f"  Political: {df_sample['is_political'].sum()}")
print(f"  General: {(~df_sample['is_political']).sum()}")

Political tweets: 611,867
General tweets: 1,168,081

Final sample size: 75000
  Political: 37500
  General: 37500


## 2. BERTopic Topic Modeling

### Improvements over baseline:
- **HDBSCAN `min_cluster_size`**: reduced from 50 to 20 to capture smaller coherent clusters
- **HDBSCAN `min_samples`**: set to 10 for better outlier boundary detection
- **UMAP `n_neighbors`**: increased from 15 to 20 for richer local structure
- **Post-hoc outlier reduction**: using BERTopic's `reduce_outliers()` method
- **Stratified sampling**: political keyword pre-filtering increases signal density

In [8]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(
    n_neighbors=20,
    n_components=5,
    metric='cosine',
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=3
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    top_n_words=10,
    verbose=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
topics, probs = topic_model.fit_transform(docs)

topic_info = topic_model.get_topic_info()
n_outliers_before = sum(1 for t in topics if t == -1)
pct_before = n_outliers_before / len(topics) * 100
print(f"Topics discovered: {len(topic_info) - 1}")
print(f"Outliers BEFORE reduction: {n_outliers_before} ({pct_before:.1f}%)")
print()
print(topic_info.head(15))

2026-05-25 08:32:49,880 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

2026-05-25 08:33:25,450 - BERTopic - Embedding - Completed ✓
2026-05-25 08:33:25,451 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-25 08:36:33,708 - BERTopic - Dimensionality - Completed ✓
2026-05-25 08:36:33,712 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-25 08:50:25,773 - BERTopic - Cluster - Completed ✓
2026-05-25 08:50:25,797 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-25 08:50:28,761 - BERTopic - Representation - Completed ✓


Topics discovered: 409
Outliers BEFORE reduction: 39288 (52.4%)

    Topic  Count                                               Name  \
0      -1  39288                             -1_like_trump_just_don   
1       0   1834                        0_israel_hamas_gaza_israeli   
2       1   1359                                1_game_team_qb_penn   
3       2    979                       2_god_christian_jesus_church   
4       3    788                         3_housing_road_parking_car   
5       4    733                     4_ukraine_russia_putin_russian   
6       5    722                5_court_judge_supreme_supreme court   
7       6    687                    6_olivia_looks_album_looks like   
8       7    679                        7_woman_questions_vp_record   
9       8    660               8_abortion_abortions_pregnant_babies   
10      9    563  9_kamala_kamala campaign_kamala just_kamala ka...   
11     10    495                          10_trans_women_men_gender   
12     11   

### Outlier Reduction

BERTopic's `reduce_outliers()` reassigns noise-cluster documents to their nearest topic based on c-TF-IDF similarity.

In [10]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf", threshold=0.1)
topic_model.update_topics(docs, topics=new_topics)

n_outliers_after = sum(1 for t in new_topics if t == -1)
pct_after = n_outliers_after / len(new_topics) * 100

print(f"Outliers BEFORE reduction: {n_outliers_before} ({pct_before:.1f}%)")
print(f"Outliers AFTER reduction:  {n_outliers_after} ({pct_after:.1f}%)")
print(f"Reduction: {n_outliers_before - n_outliers_after} tweets reassigned")

df_sample['topic'] = new_topics

topic_info_updated = topic_model.get_topic_info()
print(f"\nUpdated topic info:")
print(topic_info_updated.head(15))

2026-05-25 08:50:34,659 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers BEFORE reduction: 39288 (52.4%)
Outliers AFTER reduction:  22482 (30.0%)
Reduction: 16806 tweets reassigned

Updated topic info:
    Topic  Count                               Name  \
0      -1  22482                  -1_it_you_he_that   
1       0   1882        0_israel_hamas_gaza_israeli   
2       1   1394      1_game_team_miamidolphins_nfl   
3       2   1056       2_god_jesus_christian_church   
4       3    808         3_car_housing_road_parking   
5       4    802     4_ukraine_russia_putin_russian   
6       5    843        5_court_supreme_judge_trial   
7       6    688           6_she_her_olivia_herself   
8       7    679             7_she_her_woman_border   
9       8    718  8_abortion_abortions_babies_birth   
10      9    575          9_kamala_she_her_campaign   
11     10    583             10_trans_women_men_gay   
12     11    567     11_vote_mail_ballots_elections   
13     12    512          12_biden_joe_president_he   
14     13    567         13_black_rac

### Model Evaluation

In [11]:
cleaned_docs = [doc.split() for doc in docs]
dictionary = Dictionary(cleaned_docs)

topics_words = []
for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id != -1:
        words = [word for word, _ in topic_model.get_topic(topic_id)]
        words = [word for word in words if word in dictionary.token2id]
        if len(words) > 0:
            topics_words.append(words)

coherence_model = CoherenceModel(
    topics=topics_words,
    texts=cleaned_docs,
    dictionary=dictionary,
    coherence="c_v"
)
coherence_score = coherence_model.get_coherence()

def topic_diversity(tw, top_n=10):
    unique_words = set()
    total_words = []
    for topic in tw:
        top_words = topic[:top_n]
        unique_words.update(top_words)
        total_words.extend(top_words)
    return len(unique_words) / len(total_words) if total_words else 0

diversity_score = topic_diversity(topics_words)

print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)
print(f"{'Metric':<25} {'Baseline':>10} {'Improved':>10}")
print("-" * 50)
print(f"{'Coherence (c_v)':<25} {'0.4609':>10} {coherence_score:>10.4f}")
print(f"{'Topic Diversity':<25} {'0.8972':>10} {diversity_score:>10.4f}")
print(f"{'Outlier %':<25} {'51.4%':>10} {pct_after:>9.1f}%")
print(f"{'Num Topics':<25} {'84':>10} {len(topics_words):>10}")
print("=" * 50)

MODEL EVALUATION
Metric                      Baseline   Improved
--------------------------------------------------
Coherence (c_v)               0.4609     0.4291
Topic Diversity               0.8972     0.7216
Outlier %                      51.4%      30.0%
Num Topics                        84        409


### Topic Labeling and Classification

We inspect the discovered topics and assign human-readable labels. Topics are classified as political or non-political for downstream analysis.

In [12]:
print("Top 20 topics by size:")
print("=" * 80)
for _, row in topic_info_updated.head(21).iterrows():
    topic_id = row['Topic']
    count = row['Count']
    if topic_id == -1:
        label = "NOISE"
    else:
        words = [w for w, _ in topic_model.get_topic(topic_id)][:5]
        label = ", ".join(words)
    print(f"  Topic {topic_id:>3}: {count:>5} tweets | {label}")

Top 20 topics by size:
  Topic  -1: 22482 tweets | NOISE
  Topic   0:  1882 tweets | israel, hamas, gaza, israeli, jews
  Topic   1:  1394 tweets | game, team, miamidolphins, nfl, season
  Topic   2:  1056 tweets | god, jesus, christian, church, christians
  Topic   3:   808 tweets | car, housing, road, parking, bike
  Topic   4:   802 tweets | ukraine, russia, putin, russian, nato
  Topic   5:   843 tweets | court, supreme, judge, trial, judges
  Topic   6:   688 tweets | she, her, olivia, herself, looks
  Topic   7:   679 tweets | she, her, woman, border, questions
  Topic   8:   718 tweets | abortion, abortions, babies, birth, pregnant
  Topic   9:   575 tweets | kamala, she, her, campaign, is
  Topic  10:   583 tweets | trans, women, men, gay, woman
  Topic  11:   567 tweets | vote, mail, ballots, elections, voting
  Topic  12:   512 tweets | biden, joe, president, he, his
  Topic  13:   567 tweets | black, racist, white, race, racism
  Topic  14:   510 tweets | gun, guns, ar, laws

In [13]:
# Map topics to political issue labels based on top words
# This mapping will need to be adjusted based on the actual discovered topics
topic_labels = {}
political_topic_ids = []

for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id == -1:
        topic_labels[topic_id] = "Noise"
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)][:10]
    words_str = " ".join(words).lower()

    label = None
    is_political = False

    if any(w in words_str for w in ['israel', 'gaza', 'hamas', 'palestinian', 'jewish']):
        label = "Middle East / Israel-Gaza"
        is_political = True
    elif any(w in words_str for w in ['tax', 'inflation', 'economy', 'prices', 'economic']):
        label = "Economic Policy"
        is_political = True
    elif any(w in words_str for w in ['healthcare', 'health', 'patient', 'medical', 'insurance']):
        label = "Healthcare"
        is_political = True
    elif any(w in words_str for w in ['immigration', 'border', 'immigrant', 'ice', 'deport', 'migrant']):
        label = "Immigration / Border"
        is_political = True
    elif any(w in words_str for w in ['kamala', 'harris', 'trump', 'biden', 'election', 'vote', 'ballot']):
        label = "2024 Election / Candidates"
        is_political = True
    elif any(w in words_str for w in ['gun', 'shooting', 'firearm', 'amendment']):
        label = "Gun Policy"
        is_political = True
    elif any(w in words_str for w in ['climate', 'environment', 'energy', 'fossil', 'carbon']):
        label = "Climate / Environment"
        is_political = True
    elif any(w in words_str for w in ['abortion', 'roe', 'reproductive', 'pro life', 'pro choice']):
        label = "Abortion / Reproductive Rights"
        is_political = True
    elif any(w in words_str for w in ['ukraine', 'russia', 'nato', 'foreign policy', 'china']):
        label = "Foreign Policy"
        is_political = True
    elif any(w in words_str for w in ['police', 'crime', 'justice', 'prison', 'law enforcement']):
        label = "Criminal Justice"
        is_political = True
    elif any(w in words_str for w in ['education', 'school', 'student', 'college', 'university']):
        label = "Education"
        is_political = True
    elif any(w in words_str for w in ['democrat', 'republican', 'gop', 'liberal', 'conservative', 'maga']):
        label = "Partisan Discourse"
        is_political = True
    elif any(w in words_str for w in ['god', 'jesus', 'church', 'faith', 'christian', 'religion']):
        label = "Religion / Faith"
        is_political = False
    elif any(w in words_str for w in ['morning', 'good morning', 'good night', 'blessed']):
        label = "General Greetings"
        is_political = False
    elif any(w in words_str for w in ['game', 'team', 'player', 'season', 'football', 'nfl', 'nba']):
        label = "Sports"
        is_political = False
    elif any(w in words_str for w in ['thank', 'follow', 'giveaway', 'retweet']):
        label = "Spam / Engagement Bait"
        is_political = False
    elif any(w in words_str for w in ['taylor', 'swift', 'concert', 'album', 'music']):
        label = "Pop Culture"
        is_political = False
    else:
        label = f"Other ({', '.join(words[:3])})"
        is_political = False

    topic_labels[topic_id] = label
    if is_political:
        political_topic_ids.append(topic_id)

df_sample['topic_label'] = df_sample['topic'].map(topic_labels)

print(f"Political topics identified: {len(political_topic_ids)}")
print(f"\nPolitical topics:")
for tid in political_topic_ids:
    count = (df_sample['topic'] == tid).sum()
    print(f"  {tid:>3}: {topic_labels[tid]:<35} ({count} tweets)")

df_political = df_sample[df_sample['topic'].isin(political_topic_ids)].copy()
print(f"\nTotal political tweets: {len(df_political)} ({len(df_political)/len(df_sample)*100:.1f}%)")

Political topics identified: 168

Political topics:
    0: Middle East / Israel-Gaza           (1882 tweets)
    4: Foreign Policy                      (802 tweets)
    7: Immigration / Border                (679 tweets)
    8: Abortion / Reproductive Rights      (718 tweets)
    9: 2024 Election / Candidates          (575 tweets)
   11: 2024 Election / Candidates          (567 tweets)
   12: 2024 Election / Candidates          (512 tweets)
   14: Gun Policy                          (510 tweets)
   15: Immigration / Border                (434 tweets)
   16: Partisan Discourse                  (457 tweets)
   17: 2024 Election / Candidates          (417 tweets)
   25: Immigration / Border                (262 tweets)
   26: 2024 Election / Candidates          (274 tweets)
   27: Climate / Environment               (274 tweets)
   28: Immigration / Border                (303 tweets)
   31: Immigration / Border                (274 tweets)
   32: Economic Policy                     (281 twe

### State-Level Topic Distributions

In [14]:
state_name_to_code = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
    "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY", "District of Columbia": "DC"
}

df_sample["state_code"] = df_sample["state"].map(state_name_to_code)

exclude_states = ["USA", "US", "United States", None, "Unknown"]
df_states = df_sample[~df_sample["state"].isin(exclude_states)].copy()
df_states = df_states.dropna(subset=["state_code"])

print(f"Tweets with valid state: {len(df_states)}")
print(f"States represented: {df_states['state_code'].nunique()}")

Tweets with valid state: 60140
States represented: 48


In [15]:
df_pol_states = df_states[df_states['topic'].isin(political_topic_ids)].copy()

topic_state_counts = (
    df_pol_states.groupby(["topic_label", "state_code"])
    .size()
    .reset_index(name="count")
)

topic_state_counts["pct"] = (
    topic_state_counts.groupby("topic_label")["count"]
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
)

fig = px.treemap(
    topic_state_counts,
    path=["topic_label", "state_code"],
    values="count",
    custom_data=["pct", "state_code"],
    title="States Grouped by Policy Issue",
    color="topic_label",
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig.update_traces(
    texttemplate="%{customdata[1]}<br>%{customdata[0]}%",
    textposition="middle center"
)
fig.update_layout(height=700)
fig.show()

In [16]:
state_topic = (
    df_pol_states.groupby(["state_code", "topic_label"])
    .size()
    .reset_index(name="count")
)
state_totals = df_pol_states.groupby("state_code").size().reset_index(name="total")
state_topic = state_topic.merge(state_totals, on="state_code")
state_topic["pct"] = (state_topic["count"] / state_topic["total"] * 100).round(1)

fig = px.bar(
    state_topic,
    x="state_code", y="pct", color="topic_label",
    title="Policy Issue Composition by State",
    labels={"pct": "% of tweets", "state_code": "State", "topic_label": "Policy Issue"},
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig.update_layout(
    barmode="stack",
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(title="% of tweets"),
    height=600,
    legend=dict(title="Policy Issue", orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(t=80, b=120, l=60, r=200)
)
fig.show()

## 3. VADER Sentiment Analysis

VADER (Valence Aware Dictionary and sEntiment Reasoner) is well-suited for social media text. It handles slang, capitalization emphasis, and emoticons without requiring labeled training data. The compound score ranges from -1 (most negative) to +1 (most positive).

In [17]:
analyzer = SentimentIntensityAnalyzer()

df_sample['vader_compound'] = df_sample['tweet_clean'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)
df_sample['vader_label'] = df_sample['vader_compound'].apply(
    lambda x: 'positive' if x >= 0.05 else ('negative' if x <= -0.05 else 'neutral')
)

print("Overall sentiment distribution:")
print(df_sample['vader_label'].value_counts())
print(f"\nMean compound score: {df_sample['vader_compound'].mean():.4f}")
print(f"Median compound score: {df_sample['vader_compound'].median():.4f}")

Overall sentiment distribution:
vader_label
positive    30209
negative    28137
neutral     16654
Name: count, dtype: int64

Mean compound score: 0.0044
Median compound score: 0.0000


### Sentiment by Political Topic

In [18]:
df_pol_sent = df_sample[df_sample['topic'].isin(political_topic_ids)].copy()

topic_sentiment = df_pol_sent.groupby('topic_label')['vader_compound'].agg(['mean', 'median', 'std', 'count']).round(4)
topic_sentiment = topic_sentiment.sort_values('mean')
print(topic_sentiment)

fig = px.bar(
    topic_sentiment.reset_index(),
    x='topic_label', y='mean',
    error_y='std',
    title="Mean VADER Sentiment by Political Topic",
    labels={"mean": "Mean Compound Score", "topic_label": "Political Topic"},
    color='mean',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0
)
fig.update_layout(xaxis_tickangle=45, height=500)
fig.show()

                                  mean  median     std  count
topic_label                                                  
Criminal Justice               -0.3509 -0.5423  0.5369    240
Middle East / Israel-Gaza      -0.2054 -0.2500  0.5153   1995
Foreign Policy                 -0.0961  0.0000  0.5489   1193
2024 Election / Candidates     -0.0955  0.0000  0.4894   6382
Abortion / Reproductive Rights -0.0945  0.0000  0.5184    783
Gun Policy                     -0.0820  0.0000  0.5796    814
Partisan Discourse             -0.0632  0.0000  0.5250   1970
Economic Policy                -0.0246  0.0000  0.5096   1270
Immigration / Border           -0.0079  0.0000  0.5070   4963
Education                       0.0057  0.0000  0.5168    517
Climate / Environment           0.1186  0.0000  0.5171    604
Healthcare                      0.3700  0.4939  0.4880   2682


### Sentiment by State

In [19]:
df_states_sent = df_states.copy()
df_states_sent['vader_compound'] = df_sample.loc[df_states_sent.index, 'vader_compound']

state_sentiment = df_states_sent.groupby('state_code')['vader_compound'].mean().reset_index()
state_sentiment.columns = ['state_code', 'mean_sentiment']

fig = px.choropleth(
    state_sentiment,
    locations='state_code',
    locationmode='USA-states',
    color='mean_sentiment',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    scope='usa',
    title='Average Tweet Sentiment by State'
)
fig.update_layout(height=500)
fig.show()

### State x Topic Sentiment Heatmap

In [20]:
df_cross = df_pol_states.copy()
df_cross['vader_compound'] = df_sample.loc[df_cross.index, 'vader_compound']

pivot = df_cross.groupby(['state_code', 'topic_label'])['vader_compound'].mean().unstack(fill_value=0)

fig = px.imshow(
    pivot,
    labels=dict(x="Political Topic", y="State", color="Sentiment"),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title="Sentiment Heatmap: State × Political Topic",
    aspect='auto'
)
fig.update_layout(height=800)
fig.show()

## 4. Zero-Shot Partisan Stance Classification

We use a pre-trained transformer model for zero-shot classification to detect partisan stance in political tweets. This captures information that neither BERTopic (what people discuss) nor VADER (emotional tone) can provide: **which political side** the tweet supports.

Due to computational constraints, we apply this to a subsample of political tweets and aggregate results at the state level.

In [21]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE
)

candidate_labels = [
    "supports Republican party",
    "supports Democratic party",
    "neutral or nonpartisan"
]

print(f"Model loaded: facebook/bart-large-mnli (device={DEVICE})")
print(f"Candidate labels: {candidate_labels}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: facebook/bart-large-mnli (device=0)
Candidate labels: ['supports Republican party', 'supports Democratic party', 'neutral or nonpartisan']


In [22]:
llm_sample_size = min(3000, len(df_political))
df_llm = df_political.sample(n=llm_sample_size, random_state=RANDOM_STATE).copy()

print(f"Classifying {len(df_llm)} political tweets...")
print("This may take 30-60 minutes on CPU.")

stance_results = []
batch_size = 8
texts = df_llm['tweet_clean'].tolist()

for i in tqdm(range(0, len(texts), batch_size), desc="Stance classification"):
    batch = texts[i:i+batch_size]
    results = classifier(batch, candidate_labels=candidate_labels, batch_size=batch_size)
    if not isinstance(results, list):
        results = [results]
    for r in results:
        stance_results.append({
            'stance_label': r['labels'][0],
            'stance_score': r['scores'][0]
        })

df_llm['stance_label'] = [r['stance_label'] for r in stance_results]
df_llm['stance_score'] = [r['stance_score'] for r in stance_results]

print(f"\nStance distribution:")
print(df_llm['stance_label'].value_counts())
print(f"\nMean confidence: {df_llm['stance_score'].mean():.3f}")

Classifying 3000 political tweets...
This may take 30-60 minutes on CPU.


Stance classification: 100%|██████████| 375/375 [01:58<00:00,  3.18it/s]


Stance distribution:
stance_label
neutral or nonpartisan       1846
supports Democratic party     666
supports Republican party     488
Name: count, dtype: int64

Mean confidence: 0.650


In [23]:
print("High-confidence examples per stance:\n")
for label in candidate_labels:
    subset = df_llm[df_llm['stance_label'] == label].nlargest(3, 'stance_score')
    print(f"--- {label.upper()} ---")
    for _, row in subset.iterrows():
        print(f"  [{row['stance_score']:.2f}] {row['tweet_clean'][:120]}")
    print()

High-confidence examples per stance:

--- SUPPORTS REPUBLICAN PARTY ---
  [1.00] Of course Brandt is a Trumper just look at him He has been MAGA ever since his last rehab
  [1.00] She s the face of the new Republican Party A morally bereft unrepentant sinner She will fit in perfectly with their new 
  [1.00] According to Republican VP nominee JD Vance the education was wasted on a woman who can no longer be a productive member

--- SUPPORTS DEMOCRATIC PARTY ---
  [0.99] Yes Project 2025 is a DEMOCRAT thing They are the only ones talking about it This guy endorses her
  [0.99] Allowing parents to sterilize children with hormones and surgery and open borders in a society that has forced taxation 
  [0.99] KOMO is lying They are paid by the DEMS to lie to you

--- NEUTRAL OR NONPARTISAN ---
  [0.99] This isn t political views this is literally just shooting the country in the foot I have friends with differing views b
  [0.99] These protesters are actively harming Palestinians who they cl

### State-Level Partisan Lean from LLM

In [24]:
df_llm['state_code'] = df_sample.loc[df_llm.index, 'state_code']
df_llm_states = df_llm.dropna(subset=['state_code'])

state_stance = df_llm_states.groupby('state_code')['stance_label'].value_counts().unstack(fill_value=0)

for col in candidate_labels:
    if col not in state_stance.columns:
        state_stance[col] = 0

state_stance['total'] = state_stance.sum(axis=1)
state_stance['pct_dem'] = state_stance['supports Democratic party'] / state_stance['total']
state_stance['pct_rep'] = state_stance['supports Republican party'] / state_stance['total']
state_stance['partisan_lean'] = state_stance['pct_dem'] - state_stance['pct_rep']

min_tweets_for_lean = 10
state_stance_valid = state_stance[state_stance['total'] >= min_tweets_for_lean].copy()

fig = px.choropleth(
    state_stance_valid.reset_index(),
    locations='state_code',
    locationmode='USA-states',
    color='partisan_lean',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    scope='usa',
    title='LLM-Derived Partisan Lean by State (Blue=Dem, Red=Rep)',
    labels={'partisan_lean': 'Partisan Lean'}
)
fig.update_layout(height=500)
fig.show()

print(f"States with sufficient data: {len(state_stance_valid)}")
print(f"\nTop 5 Democratic-leaning states:")
print(state_stance_valid.nlargest(5, 'partisan_lean')[['partisan_lean', 'total']])
print(f"\nTop 5 Republican-leaning states:")
print(state_stance_valid.nsmallest(5, 'partisan_lean')[['partisan_lean', 'total']])

States with sufficient data: 30

Top 5 Democratic-leaning states:
stance_label  partisan_lean  total
state_code                        
TN                 0.447761    134
OH                 0.271605     81
NV                 0.263158     57
SC                 0.181818     11
CT                 0.166667     12

Top 5 Republican-leaning states:
stance_label  partisan_lean  total
state_code                        
OR                -0.266667     30
AZ                -0.189189     37
LA                -0.142857     21
MN                -0.125000     24
ID                -0.100000     10


## 5. Feature Engineering and Election Data

We construct a state-level feature matrix combining outputs from all three methods (BERTopic, VADER, Zero-Shot Stance) with engagement metrics. The target variable is the 2024 presidential election outcome by state.

In [25]:
election_2024 = {
    'AL': ('R', 25.2), 'AK': ('R', 14.9), 'AZ': ('R', 5.5), 'AR': ('R', 28.3),
    'CA': ('D', -19.5), 'CO': ('D', -11.4), 'CT': ('D', -12.4), 'DE': ('D', -11.0),
    'DC': ('D', -75.0), 'FL': ('R', 13.2), 'GA': ('R', 2.2), 'HI': ('D', -19.1),
    'ID': ('R', 32.3), 'IL': ('D', -7.5), 'IN': ('R', 20.0), 'IA': ('R', 13.5),
    'KS': ('R', 14.5), 'KY': ('R', 25.6), 'LA': ('R', 19.7), 'ME': ('D', -6.3),
    'MD': ('D', -21.0), 'MA': ('D', -23.2), 'MI': ('R', 1.4), 'MN': ('D', -2.3),
    'MS': ('R', 18.6), 'MO': ('R', 17.0), 'MT': ('R', 18.5), 'NE': ('R', 17.5),
    'NV': ('R', 3.2), 'NH': ('D', -1.2), 'NJ': ('D', -5.7), 'NM': ('D', -3.0),
    'NY': ('D', -12.2), 'NC': ('R', 3.3), 'ND': ('R', 27.8), 'OH': ('R', 11.1),
    'OK': ('R', 33.7), 'OR': ('D', -6.1), 'PA': ('R', 1.7), 'RI': ('D', -10.0),
    'SC': ('R', 14.5), 'SD': ('R', 22.0), 'TN': ('R', 25.8), 'TX': ('R', 13.6),
    'UT': ('R', 18.8), 'VT': ('D', -22.8), 'VA': ('D', -3.7), 'WA': ('D', -12.2),
    'WV': ('R', 32.0), 'WI': ('R', 0.9), 'WY': ('R', 43.0)
}

df_election = pd.DataFrame([
    {'state_code': k, 'winner': v[0], 'margin_r': v[1]}
    for k, v in election_2024.items()
])
df_election['outcome_binary'] = (df_election['winner'] == 'R').astype(int)

print(f"States: {len(df_election)}")
print(f"Republican wins: {(df_election['winner'] == 'R').sum()}")
print(f"Democratic wins: {(df_election['winner'] == 'D').sum()}")

States: 51
Republican wins: 31
Democratic wins: 20


### Build State-Level Feature Matrix

In [26]:
features = {}

# 1. Topic distribution per state (% of political tweets in each topic)
for state in df_states['state_code'].unique():
    state_data = df_states[df_states['state_code'] == state]
    state_pol = state_data[state_data['topic'].isin(political_topic_ids)]
    total = len(state_pol) if len(state_pol) > 0 else 1

    feat = {'state_code': state, 'tweet_count': len(state_data), 'political_tweet_count': len(state_pol)}

    for tid in political_topic_ids:
        label = topic_labels[tid].replace(" ", "_").replace("/", "_").lower()
        feat[f'topic_pct_{label}'] = (state_pol['topic'] == tid).sum() / total * 100

    # 2. VADER sentiment
    state_sent = df_sample.loc[state_data.index, 'vader_compound']
    feat['sentiment_mean'] = state_sent.mean() if len(state_sent) > 0 else 0
    feat['sentiment_std'] = state_sent.std() if len(state_sent) > 1 else 0

    # 3. Engagement metrics
    for metric in ['public_metrics.like_count', 'public_metrics.retweet_count.tweets_historical',
                    'public_metrics.impression_count.tweets_historical']:
        if metric in state_data.columns:
            col_clean = metric.split('.')[-1].replace('.', '_')
            vals = pd.to_numeric(state_data[metric], errors='coerce')
            feat[f'eng_{col_clean}'] = vals.mean() if not vals.isna().all() else 0

    features[state] = feat

df_features = pd.DataFrame(features.values())

# 4. Add LLM partisan lean
lean_data = state_stance_valid[['partisan_lean']].reset_index()
df_features = df_features.merge(lean_data, on='state_code', how='left')
df_features['partisan_lean'] = df_features['partisan_lean'].fillna(0)

# 5. Merge with election data
df_model = df_features.merge(df_election, on='state_code', how='inner')

# Remove states with very few tweets
min_tweets = 30
df_model = df_model[df_model['tweet_count'] >= min_tweets]
print(f"States in model (>={min_tweets} tweets): {len(df_model)}")
print(f"  Republican: {(df_model['outcome_binary'] == 1).sum()}")
print(f"  Democrat: {(df_model['outcome_binary'] == 0).sum()}")

excluded = set(df_election['state_code']) - set(df_model['state_code'])
if excluded:
    print(f"  Excluded (low tweet count): {excluded}")

States in model (>=30 tweets): 46
  Republican: 27
  Democrat: 19
  Excluded (low tweet count): {'WY', 'ND', 'WV', 'DE', 'AK'}


In [27]:
feature_cols = [c for c in df_model.columns if c not in
    ['state_code', 'winner', 'margin_r', 'outcome_binary', 'tweet_count', 'political_tweet_count']]

corr = df_model[feature_cols + ['outcome_binary']].corr()['outcome_binary'].drop('outcome_binary').sort_values()

fig = px.bar(
    x=corr.values, y=corr.index,
    orientation='h',
    title="Feature Correlation with Republican Win (outcome_binary)",
    labels={"x": "Pearson Correlation", "y": "Feature"},
    color=corr.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0
)
fig.update_layout(height=max(400, len(corr) * 25), yaxis=dict(tickfont=dict(size=9)))
fig.show()

## 6. Predictive Modeling

We predict state-level partisan outcomes (Republican vs. Democrat) using the feature matrix constructed from BERTopic topic distributions, VADER sentiment, LLM partisan lean, and engagement metrics.

Due to the small sample size (~45-50 states), we use Leave-One-Out Cross-Validation (LOOCV) for robust evaluation.

In [28]:
X = df_model[feature_cols].fillna(0)
y = df_model['outcome_binary']

X = X.replace([np.inf, -np.inf], 0)

loo = LeaveOneOut()

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.1, random_state=RANDOM_STATE))
    ])
}

results = {}
for name, model in models.items():
    y_pred = cross_val_predict(model, X, y, cv=loo)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'f1_weighted': f1, 'predictions': y_pred}
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.3f}")
    print(f"  F1 (weighted): {f1:.3f}")
    print(f"  Confusion Matrix:")
    print(f"  {confusion_matrix(y, y_pred)}")


Logistic Regression:
  Accuracy: 0.522
  F1 (weighted): 0.501
  Confusion Matrix:
  [[ 5 14]
 [ 8 19]]

Random Forest:
  Accuracy: 0.500
  F1 (weighted): 0.458
  Confusion Matrix:
  [[ 3 16]
 [ 7 20]]

Gradient Boosting:
  Accuracy: 0.543
  F1 (weighted): 0.518
  Confusion Matrix:
  [[ 5 14]
 [ 7 20]]


In [29]:
print("=" * 60)
print("MODEL COMPARISON (LOOCV)")
print("=" * 60)
print(f"{'Model':<25} {'Accuracy':>10} {'F1 (wtd)':>10}")
print("-" * 60)
for name, res in results.items():
    print(f"{name:<25} {res['accuracy']:>10.3f} {res['f1_weighted']:>10.3f}")
print("=" * 60)

best_name = max(results, key=lambda k: results[k]['accuracy'])
print(f"\nBest model: {best_name} (Accuracy: {results[best_name]['accuracy']:.3f})")

MODEL COMPARISON (LOOCV)
Model                       Accuracy   F1 (wtd)
------------------------------------------------------------
Logistic Regression            0.522      0.501
Random Forest                  0.500      0.458
Gradient Boosting              0.543      0.518

Best model: Gradient Boosting (Accuracy: 0.543)


### Feature Importance

In [30]:
best_model = models[best_name]
best_model.fit(X, y)

if hasattr(best_model.named_steps['clf'], 'feature_importances_'):
    importances = best_model.named_steps['clf'].feature_importances_
elif hasattr(best_model.named_steps['clf'], 'coef_'):
    importances = np.abs(best_model.named_steps['clf'].coef_[0])
else:
    importances = np.zeros(len(feature_cols))

feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=True)

fig = px.bar(
    feat_imp.tail(15),
    x='importance', y='feature',
    orientation='h',
    title=f'Top 15 Feature Importances ({best_name})',
    labels={'importance': 'Importance', 'feature': 'Feature'}
)
fig.update_layout(height=500)
fig.show()

### Prediction Map

In [31]:
best_preds = results[best_name]['predictions']
df_model['predicted'] = best_preds
df_model['correct'] = (df_model['outcome_binary'] == df_model['predicted']).astype(int)

df_model['pred_label'] = df_model['predicted'].map({1: 'R', 0: 'D'})
df_model['result_label'] = df_model.apply(
    lambda r: f"Actual: {r['winner']}, Pred: {r['pred_label']}" +
              (" ✓" if r['correct'] else " ✗"), axis=1
)

df_model['color_val'] = df_model.apply(
    lambda r: 2 if r['correct'] and r['winner'] == 'R'
    else (0 if r['correct'] and r['winner'] == 'D'
    else 1), axis=1
)

fig = px.choropleth(
    df_model,
    locations='state_code',
    locationmode='USA-states',
    color='color_val',
    color_continuous_scale=['blue', 'yellow', 'red'],
    scope='usa',
    hover_data=['state_code', 'winner', 'pred_label', 'correct'],
    title=f'Predicted vs Actual 2024 Election Outcomes ({best_name})'
)
fig.update_layout(
    height=500,
    coloraxis_showscale=False
)
fig.show()

misclassified = df_model[df_model['correct'] == 0]
print(f"Misclassified states ({len(misclassified)}):")
for _, row in misclassified.iterrows():
    print(f"  {row['state_code']}: Actual={row['winner']}, Predicted={row['pred_label']}")

Misclassified states (21):
  VA: Actual=D, Predicted=R
  NJ: Actual=D, Predicted=R
  AZ: Actual=R, Predicted=D
  NY: Actual=D, Predicted=R
  DC: Actual=D, Predicted=R
  RI: Actual=D, Predicted=R
  MA: Actual=D, Predicted=R
  FL: Actual=R, Predicted=D
  IL: Actual=D, Predicted=R
  KS: Actual=R, Predicted=D
  SC: Actual=R, Predicted=D
  NM: Actual=D, Predicted=R
  CT: Actual=D, Predicted=R
  OR: Actual=D, Predicted=R
  ME: Actual=D, Predicted=R
  MN: Actual=D, Predicted=R
  IA: Actual=R, Predicted=D
  IN: Actual=R, Predicted=D
  NH: Actual=D, Predicted=R
  VT: Actual=D, Predicted=R
  OK: Actual=R, Predicted=D


## 7. Systematic Bias Audit

We audit our pipeline for four types of bias:
1. **Geographic bias** — Are some states over/under-represented?
2. **Engagement bias** — Do high-follower accounts distort the signal?
3. **Temporal bias** — Is the sample skewed toward specific time periods?
4. **Model fairness** — Does prediction accuracy vary across state groups?

### 7.1 Geographic Bias

In [32]:
state_population_2020 = {
    'CA': 39538, 'TX': 29146, 'FL': 21538, 'NY': 20202, 'PA': 13002,
    'IL': 12812, 'OH': 11800, 'GA': 10712, 'NC': 10439, 'MI': 10078,
    'NJ': 9289, 'VA': 8632, 'WA': 7615, 'AZ': 7151, 'MA': 7030,
    'TN': 6910, 'IN': 6786, 'MD': 6178, 'MO': 6155, 'WI': 5894,
    'CO': 5774, 'MN': 5707, 'SC': 5119, 'AL': 5024, 'LA': 4658,
    'KY': 4506, 'OR': 4238, 'OK': 3960, 'CT': 3606, 'UT': 3272,
    'IA': 3190, 'NV': 3105, 'AR': 3012, 'MS': 2962, 'KS': 2937,
    'NM': 2118, 'NE': 1962, 'ID': 1901, 'WV': 1794, 'HI': 1456,
    'NH': 1378, 'ME': 1362, 'MT': 1085, 'RI': 1098, 'DE': 990,
    'SD': 887, 'ND': 779, 'AK': 733, 'DC': 690, 'VT': 643, 'WY': 577
}

total_pop = sum(state_population_2020.values())

tweet_counts = df_states.groupby('state_code').size().reset_index(name='tweets')
total_tweets = tweet_counts['tweets'].sum()

tweet_counts['pop'] = tweet_counts['state_code'].map(state_population_2020)
tweet_counts = tweet_counts.dropna(subset=['pop'])
tweet_counts['pct_tweets'] = tweet_counts['tweets'] / total_tweets * 100
tweet_counts['pct_pop'] = tweet_counts['pop'] / total_pop * 100
tweet_counts['representation_ratio'] = tweet_counts['pct_tweets'] / tweet_counts['pct_pop']

tweet_counts = tweet_counts.sort_values('representation_ratio', ascending=False)

print("Geographic Representation Analysis")
print("=" * 60)
print(f"{'State':<6} {'Tweets':>7} {'%Tweets':>8} {'%Pop':>6} {'Ratio':>7}")
print("-" * 60)
for _, row in tweet_counts.iterrows():
    flag = " ⚠" if row['representation_ratio'] < 0.3 or row['representation_ratio'] > 3.0 else ""
    print(f"{row['state_code']:<6} {int(row['tweets']):>7} {row['pct_tweets']:>7.1f}% {row['pct_pop']:>5.1f}% {row['representation_ratio']:>6.2f}{flag}")

fig = px.choropleth(
    tweet_counts,
    locations='state_code',
    locationmode='USA-states',
    color='representation_ratio',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=1.0,
    scope='usa',
    title='Geographic Representation Ratio (1.0 = proportional)',
    labels={'representation_ratio': 'Ratio'}
)
fig.update_layout(height=500)
fig.show()

Geographic Representation Analysis
State   Tweets  %Tweets   %Pop   Ratio
------------------------------------------------------------
DC        1248     2.1%   0.2%   9.97 ⚠
NV        1251     2.1%   0.9%   2.22
FL        8290    13.8%   6.5%   2.12
NJ        3515     5.8%   2.8%   2.09
HI         546     0.9%   0.4%   2.07
VT         215     0.4%   0.2%   1.84
CO        1759     2.9%   1.7%   1.68
CA       10657    17.7%  11.9%   1.49
WA        1866     3.1%   2.3%   1.35
TN        1616     2.7%   2.1%   1.29
MT         250     0.4%   0.3%   1.27
NY        4409     7.3%   6.1%   1.20
OR         860     1.4%   1.3%   1.12
AZ        1407     2.3%   2.2%   1.08
KS         565     0.9%   0.9%   1.06
MD        1113     1.9%   1.9%   0.99
OH        2116     3.5%   3.6%   0.99
MA        1080     1.8%   2.1%   0.85
PA        1918     3.2%   3.9%   0.81
MN         837     1.4%   1.7%   0.81
VA        1255     2.1%   2.6%   0.80
WI         845     1.4%   1.8%   0.79
ME         193     0.3%   0

### 7.2 Engagement Bias (Power Users)

In [33]:
if 'public_metrics.followers_count' in df_sample.columns:
    followers = pd.to_numeric(df_sample['public_metrics.followers_count'], errors='coerce')
    threshold = followers.quantile(0.99)

    df_sample['is_power_user'] = followers >= threshold

    print(f"Power user threshold (top 1%): {threshold:.0f} followers")
    print(f"Power users: {df_sample['is_power_user'].sum()}")
    print(f"Regular users: {(~df_sample['is_power_user']).sum()}")

    power_topics = df_sample[df_sample['is_power_user'] & df_sample['topic'].isin(political_topic_ids)]
    regular_topics = df_sample[~df_sample['is_power_user'] & df_sample['topic'].isin(political_topic_ids)]

    if len(power_topics) > 0 and len(regular_topics) > 0:
        power_dist = power_topics['topic_label'].value_counts(normalize=True).sort_index()
        regular_dist = regular_topics['topic_label'].value_counts(normalize=True).sort_index()

        comparison = pd.DataFrame({'power_users': power_dist, 'regular_users': regular_dist}).fillna(0)
        comparison['difference'] = comparison['power_users'] - comparison['regular_users']

        print("\nTopic distribution comparison:")
        print(comparison.round(3))

        fig = go.Figure()
        fig.add_trace(go.Bar(name='Power Users (top 1%)', x=comparison.index, y=comparison['power_users']))
        fig.add_trace(go.Bar(name='Regular Users', x=comparison.index, y=comparison['regular_users']))
        fig.update_layout(
            barmode='group',
            title='Topic Distribution: Power Users vs Regular Users',
            xaxis_tickangle=45, height=500
        )
        fig.show()
else:
    print("Followers column not found — skipping engagement bias analysis.")
    df_sample['is_power_user'] = False

Power user threshold (top 1%): 229686 followers
Power users: 930
Regular users: 74070

Topic distribution comparison:
                                power_users  regular_users  difference
topic_label                                                           
2024 Election / Candidates            0.324          0.272       0.052
Abortion / Reproductive Rights        0.049          0.033       0.016
Climate / Environment                 0.014          0.026      -0.012
Criminal Justice                      0.003          0.010      -0.008
Economic Policy                       0.046          0.054      -0.008
Education                             0.041          0.022       0.019
Foreign Policy                        0.016          0.052      -0.035
Gun Policy                            0.011          0.035      -0.024
Healthcare                            0.076          0.115      -0.039
Immigration / Border                  0.294          0.211       0.084
Middle East / Israel-Gaza     

### 7.3 Temporal Bias

In [34]:
time_col = None
for col in ['created_at', 'created_at.tweets', 'sampling_date']:
    if col in df_sample.columns:
        time_col = col
        break

if time_col:
    df_sample['datetime'] = pd.to_datetime(df_sample[time_col], errors='coerce')
    df_time = df_sample.dropna(subset=['datetime'])

    weekly = df_time.set_index('datetime').resample('W').size().reset_index(name='count')

    fig = px.line(
        weekly, x='datetime', y='count',
        title='Tweet Volume Over Time (Weekly)',
        labels={'datetime': 'Date', 'count': 'Tweets per Week'}
    )
    fig.update_layout(height=400)
    fig.show()

    print(f"Date range: {df_time['datetime'].min()} to {df_time['datetime'].max()}")
    print(f"Tweets with valid timestamp: {len(df_time)}")
else:
    print("No timestamp column found — skipping temporal bias analysis.")

Date range: 2008-04-04 04:02:42+00:00 to 2024-10-30 21:11:00+00:00
Tweets with valid timestamp: 75000


### 7.4 Model Fairness Across State Groups

In [35]:
swing_states = ['AZ', 'GA', 'MI', 'NV', 'NC', 'PA', 'WI']
df_model['is_swing'] = df_model['state_code'].isin(swing_states)

regions = {
    'Northeast': ['CT', 'DE', 'ME', 'MD', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT', 'DC'],
    'South': ['AL', 'AR', 'FL', 'GA', 'KY', 'LA', 'MS', 'NC', 'SC', 'TN', 'TX', 'VA', 'WV'],
    'Midwest': ['IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI'],
    'West': ['AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY']
}
state_to_region = {s: r for r, states in regions.items() for s in states}
df_model['region'] = df_model['state_code'].map(state_to_region)

print("Model Fairness Analysis")
print("=" * 60)

# Swing vs Safe
swing = df_model[df_model['is_swing']]
safe = df_model[~df_model['is_swing']]
if len(swing) > 0:
    print(f"\nSwing states accuracy: {swing['correct'].mean():.3f} (n={len(swing)})")
if len(safe) > 0:
    print(f"Safe states accuracy:  {safe['correct'].mean():.3f} (n={len(safe)})")

# By region
print(f"\nAccuracy by region:")
for region in ['Northeast', 'South', 'Midwest', 'West']:
    subset = df_model[df_model['region'] == region]
    if len(subset) > 0:
        print(f"  {region:<12}: {subset['correct'].mean():.3f} (n={len(subset)})")

Model Fairness Analysis

Swing states accuracy: 0.857 (n=7)
Safe states accuracy:  0.487 (n=39)

Accuracy by region:
  Northeast   : 0.182 (n=11)
  South       : 0.750 (n=12)
  Midwest     : 0.545 (n=11)
  West        : 0.727 (n=11)


### 7.5 Audited Model: Impact of Bias Reduction

In [36]:
# Re-run best model without power users
if df_sample['is_power_user'].any():
    df_no_power = df_sample[~df_sample['is_power_user']].copy()
    df_no_power_states = df_no_power[~df_no_power["state"].isin(exclude_states)].dropna(subset=["state_code"])

    features_audited = {}
    for state in df_no_power_states['state_code'].unique():
        state_data = df_no_power_states[df_no_power_states['state_code'] == state]
        state_pol = state_data[state_data['topic'].isin(political_topic_ids)]
        total = len(state_pol) if len(state_pol) > 0 else 1

        feat = {'state_code': state, 'tweet_count': len(state_data)}

        for tid in political_topic_ids:
            label = topic_labels[tid].replace(" ", "_").replace("/", "_").lower()
            feat[f'topic_pct_{label}'] = (state_pol['topic'] == tid).sum() / total * 100

        state_sent = df_no_power.loc[state_data.index, 'vader_compound'] if 'vader_compound' in df_no_power.columns else pd.Series([0])
        feat['sentiment_mean'] = state_sent.mean() if len(state_sent) > 0 else 0
        feat['sentiment_std'] = state_sent.std() if len(state_sent) > 1 else 0
        feat['partisan_lean'] = state_stance_valid.loc[state, 'partisan_lean'] if state in state_stance_valid.index else 0

        features_audited[state] = feat

    df_feat_audited = pd.DataFrame(features_audited.values())
    df_model_audited = df_feat_audited.merge(df_election, on='state_code', how='inner')
    df_model_audited = df_model_audited[df_model_audited['tweet_count'] >= min_tweets]

    audit_feature_cols = [c for c in df_model_audited.columns if c not in
        ['state_code', 'winner', 'margin_r', 'outcome_binary', 'tweet_count']]

    X_audit = df_model_audited[audit_feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
    y_audit = df_model_audited['outcome_binary']

    if len(X_audit) > 5:
        best_audit = models[best_name]
        y_pred_audit = cross_val_predict(best_audit, X_audit, y_audit, cv=LeaveOneOut())
        acc_audit = accuracy_score(y_audit, y_pred_audit)

        print("=" * 60)
        print("BIAS AUDIT: MODEL COMPARISON")
        print("=" * 60)
        print(f"{'Metric':<30} {'Baseline':>10} {'Audited':>10}")
        print("-" * 60)
        print(f"{'Accuracy':<30} {results[best_name]['accuracy']:>10.3f} {acc_audit:>10.3f}")
        print(f"{'States used':<30} {len(df_model):>10} {len(df_model_audited):>10}")
        print(f"{'Power users removed':<30} {'No':>10} {'Yes':>10}")
        print("=" * 60)
    else:
        print("Not enough states after filtering for audited model comparison.")
else:
    print("No power users identified — audited model same as baseline.")

BIAS AUDIT: MODEL COMPARISON
Metric                           Baseline    Audited
------------------------------------------------------------
Accuracy                            0.543      0.587
States used                            46         46
Power users removed                    No        Yes


## 8. Summary

In [37]:
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print()
print("TOPIC MODELING (BERTopic)")
print(f"  Topics discovered: {len(topics_words)}")
print(f"  Coherence (c_v):   {coherence_score:.4f} (baseline: 0.4609)")
print(f"  Topic diversity:   {diversity_score:.4f} (baseline: 0.8972)")
print(f"  Outlier rate:      {pct_after:.1f}% (baseline: 51.4%)")
print(f"  Political topics:  {len(political_topic_ids)}")
print()
print("SENTIMENT ANALYSIS (VADER)")
print(f"  Mean compound:     {df_sample['vader_compound'].mean():.4f}")
print(f"  Positive:          {(df_sample['vader_label']=='positive').sum()}")
print(f"  Neutral:           {(df_sample['vader_label']=='neutral').sum()}")
print(f"  Negative:          {(df_sample['vader_label']=='negative').sum()}")
print()
print("STANCE DETECTION (Zero-Shot LLM)")
print(f"  Tweets classified: {len(df_llm)}")
print(f"  Mean confidence:   {df_llm['stance_score'].mean():.3f}")
for label in candidate_labels:
    count = (df_llm['stance_label'] == label).sum()
    print(f"  {label}: {count}")
print()
print("PREDICTIVE MODELING")
print(f"  States in model:   {len(df_model)}")
print(f"  Best model:        {best_name}")
print(f"  Accuracy (LOOCV):  {results[best_name]['accuracy']:.3f}")
print(f"  F1 (weighted):     {results[best_name]['f1_weighted']:.3f}")
print(f"  Misclassified:     {len(misclassified)} states")
print("=" * 70)

FINAL RESULTS SUMMARY

TOPIC MODELING (BERTopic)
  Topics discovered: 409
  Coherence (c_v):   0.4291 (baseline: 0.4609)
  Topic diversity:   0.7216 (baseline: 0.8972)
  Outlier rate:      30.0% (baseline: 51.4%)
  Political topics:  168

SENTIMENT ANALYSIS (VADER)
  Mean compound:     0.0044
  Positive:          30209
  Neutral:           16654
  Negative:          28137

STANCE DETECTION (Zero-Shot LLM)
  Tweets classified: 3000
  Mean confidence:   0.650
  supports Republican party: 488
  supports Democratic party: 666
  neutral or nonpartisan: 1846

PREDICTIVE MODELING
  States in model:   46
  Best model:        Gradient Boosting
  Accuracy (LOOCV):  0.543
  F1 (weighted):     0.518
  Misclassified:     21 states


### Limitations

1. **Sample size**: Only ~50 states as prediction units — inherently limits model complexity and generalizability.
2. **Twitter/X representativeness**: The platform's user base skews younger, more urban, and more politically engaged than the general electorate.
3. **Geographic coverage**: Small states may have insufficient tweet volume for reliable feature estimation.
4. **Temporal scope**: Tweets from a single election cycle; patterns may not generalize across cycles.
5. **Zero-shot classification**: Pre-trained model may carry biases from its training data. No domain-specific fine-tuning was performed.
6. **Keyword filtering**: Political keyword pre-filtering may miss subtle political content or capture false positives.
7. **State attribution**: State labels derived from user profiles may not reflect where users actually vote.

## 9. Export Artifacts for Streamlit App

Save all trained models and pre-computed data, then upload to HuggingFace so the Streamlit app can load them.

**Two options:**
- **Option A**: Upload to HuggingFace Hub (for HF Spaces deployment)
- **Option B**: Save locally / to Google Drive

In [38]:
import joblib
import os

EXPORT_DIR = "app_artifacts"
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Save BERTopic model
topic_model.save(os.path.join(EXPORT_DIR, "topic_model"))
print("Saved: topic_model/")

# 2. Save best predictive model
best_model.fit(X, y)
joblib.dump(best_model, os.path.join(EXPORT_DIR, "best_model.pkl"))
print(f"Saved: best_model.pkl ({best_name})")

# 3. Save DataFrames
export_cols_sample = ['tweet_clean', 'topic', 'topic_label', 'vader_compound',
                      'vader_label', 'state_code', 'is_political']
if 'is_power_user' in df_sample.columns:
    export_cols_sample.append('is_power_user')
available_cols = [c for c in export_cols_sample if c in df_sample.columns]
df_sample[available_cols].to_parquet(os.path.join(EXPORT_DIR, "df_sample.parquet"), index=False)
print(f"Saved: df_sample.parquet ({len(df_sample)} rows)")

df_model.to_parquet(os.path.join(EXPORT_DIR, "df_model.parquet"), index=False)
print(f"Saved: df_model.parquet ({len(df_model)} rows)")

feat_imp.to_parquet(os.path.join(EXPORT_DIR, "feat_imp.parquet"), index=False)
print("Saved: feat_imp.parquet")

# 4. Save metadata as JSON
with open(os.path.join(EXPORT_DIR, "topic_labels.json"), "w") as f:
    json.dump({str(k): v for k, v in topic_labels.items()}, f, indent=2)

with open(os.path.join(EXPORT_DIR, "political_topic_ids.json"), "w") as f:
    json.dump(political_topic_ids, f)

model_metrics = {
    "coherence": round(coherence_score, 4),
    "diversity": round(diversity_score, 4),
    "outlier_pct": round(pct_after, 1),
    "num_topics": len(topics_words),
    "best_model": best_name,
    "best_accuracy": round(results[best_name]['accuracy'], 3),
    "best_f1": round(results[best_name]['f1_weighted'], 3),
    "model_results": {
        name: {"accuracy": round(r['accuracy'], 3), "f1_weighted": round(r['f1_weighted'], 3)}
        for name, r in results.items()
    }
}
with open(os.path.join(EXPORT_DIR, "model_metrics.json"), "w") as f:
    json.dump(model_metrics, f, indent=2)

state_meta = {
    "swing_states": ["AZ", "GA", "MI", "NV", "NC", "PA", "WI"],
    "regions": {
        "Northeast": ["CT", "DE", "ME", "MD", "MA", "NH", "NJ", "NY", "PA", "RI", "VT", "DC"],
        "South": ["AL", "AR", "FL", "GA", "KY", "LA", "MS", "NC", "SC", "TN", "TX", "VA", "WV"],
        "Midwest": ["IL", "IN", "IA", "KS", "MI", "MN", "MO", "NE", "ND", "OH", "SD", "WI"],
        "West": ["AK", "AZ", "CA", "CO", "HI", "ID", "MT", "NV", "NM", "OR", "UT", "WA", "WY"]
    },
    "election_2024": {k: str(v) for k, v in election_2024.items()}
}
with open(os.path.join(EXPORT_DIR, "state_metadata.json"), "w") as f:
    json.dump(state_meta, f, indent=2)

print("\nAll JSON metadata saved.")
print(f"\nArtifacts directory contents:")
for root, dirs, files in os.walk(EXPORT_DIR):
    for fname in files:
        fpath = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {os.path.relpath(fpath, EXPORT_DIR)}: {size_mb:.1f} MB")

2026-05-25 08:54:40,178 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saved: topic_model/
Saved: best_model.pkl (Gradient Boosting)
Saved: df_sample.parquet (75000 rows)
Saved: df_model.parquet (46 rows)
Saved: feat_imp.parquet

All JSON metadata saved.

Artifacts directory contents:
  political_topic_ids.json: 0.0 MB
  model_metrics.json: 0.0 MB
  feat_imp.parquet: 0.0 MB
  state_metadata.json: 0.0 MB
  topic_model: 761.0 MB
  best_model.pkl: 0.0 MB
  df_model.parquet: 0.0 MB
  topic_labels.json: 0.0 MB
  df_sample.parquet: 6.7 MB


### Option A: Upload artifacts to HuggingFace Hub

This uploads the artifacts to a HuggingFace dataset repo so the Streamlit Space can download them at startup. Run the cell below and follow the login prompt.

In [40]:
from huggingface_hub import HfApi, login

# Login to HuggingFace (will prompt for token)
# Get your token from: https://huggingface.co/settings/tokens
login()

api = HfApi()

# Create or use existing repo for artifacts
REPO_ID = "Diogo2110/issue-salience-artifacts"  # Change if needed
REPO_TYPE = "dataset"

try:
    api.create_repo(repo_id=REPO_ID, repo_type=REPO_TYPE, exist_ok=True)
    print(f"Repo ready: {REPO_ID}")
except Exception as e:
    print(f"Repo creation note: {e}")

# Upload all artifacts
api.upload_folder(
    folder_path=EXPORT_DIR,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
)

print(f"\nAll artifacts uploaded to: https://huggingface.co/datasets/{REPO_ID}")
print("The Streamlit app will download these at startup.")

Repo ready: Diogo2110/issue-salience-artifacts


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...app_artifacts/topic_model:   1%|          | 7.69MB /  798MB            

  ..._artifacts/best_model.pkl:   7%|7         | 3.63kB / 48.8kB            

  ...rtifacts/df_model.parquet:   7%|7         | 1.76kB / 23.7kB            

  ...tifacts/df_sample.parquet:   7%|7         |  522kB / 7.03MB            

  ...rtifacts/feat_imp.parquet:   7%|7         |   164B / 2.22kB            


All artifacts uploaded to: https://huggingface.co/datasets/Diogo2110/issue-salience-artifacts
The Streamlit app will download these at startup.


### Option B: Save to Google Drive (Colab only)

If you prefer to download the artifacts manually instead of using HuggingFace Hub.

In [ ]:
# Mount Google Drive and copy artifacts
try:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    drive_dest = '/content/drive/MyDrive/issue_salience_artifacts'
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(EXPORT_DIR, drive_dest)
    print(f"Artifacts saved to Google Drive: {drive_dest}")
except ImportError:
    print("Not running on Colab. Artifacts are saved locally in:", EXPORT_DIR)
    print("You can zip and download them:")
    print(f"  !zip -r artifacts.zip {EXPORT_DIR}")